# Lab 7 – Noun Phrase Chunking with an HMM
HMM (implemented from scratch with NumPy) that tags each word as `B-NP`, `I-NP` or `O`.

1. **Forward algorithm** → probability of each test sentence  
2. **Viterbi decoding** → predicted tag sequence, evaluated with `seqeval`

Zero counts are handled with **Add-K smoothing**.

In [1]:
try:
    import seqeval
except ImportError:
    %pip install -q seqeval

In [2]:
import os, glob
import numpy as np
from collections import Counter
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

# ---- Configuration -------------------------------------------------------
TRAIN_FILE = "train_NP.txt"   # path to the training file
TEST_FILE  = "test_NP.txt"    # path to the test file
K          = 0.1         # Add-K smoothing constant (best of the sweep in section 5)
UNK        = "<UNK>"

## 1. Load data
Expected format (CoNLL style): one token per line, the **first column is the word** and the
**last column is the tag**; a blank line separates sentences. This works with `word tag`
and `word POS tag` layouts.

In [3]:
def load_conll(path):
    sents, words, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if words:
                    sents.append((words, tags)); words, tags = [], []
                continue
            parts = line.split()
            if parts[0] == "-DOCSTART-":
                continue
            words.append(parts[0]); tags.append(parts[-1])
    if words:
        sents.append((words, tags))
    return sents

train, test = load_conll(TRAIN_FILE), load_conll(TEST_FILE)
print(f"train: {len(train)} sentences | test: {len(test)} sentences")
print(train[0])

train: 823 sentences | test: 77 sentences
(['Rockwell', 'International', 'Corp.', "'s", 'Tulsa', 'unit', 'said', 'it', 'signed', 'a', 'tentative', 'agreement', 'extending', 'its', 'contract', 'with', 'Boeing', 'Co.', 'to', 'provide', 'structural', 'parts', 'for', 'Boeing', "'s", '747', 'jetliners', '.'], ['B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'B-NP', 'I-NP', 'I-NP', 'O'])


## 2. Train the HMM (counts + Add-K smoothing)

- Initial: `π[t] = (C(start→t) + K) / (#sentences + K·N)`
- Transition: `A[i,j] = (C(i→j) + K) / (C(i) + K·N)`
- Emission: `B[t,w] = (C(t,w) + K) / (C(t) + K·(V+1))` – the extra `+1` is an `<UNK>` word so unseen test words get a non-zero probability.

All probabilities are stored/used as **logs** to avoid underflow.

In [4]:
class HMM:
    def __init__(self, k=0.01):
        self.k = k

    def fit(self, sentences):
        self.tags  = sorted({t for _, ts in sentences for t in ts})
        self.vocab = sorted({w.lower() for ws, _ in sentences for w in ws}) + [UNK]
        self.t2i = {t: i for i, t in enumerate(self.tags)}
        self.w2i = {w: i for i, w in enumerate(self.vocab)}
        N, V, k = len(self.tags), len(self.vocab), self.k

        init, trans, emit = np.zeros(N), np.zeros((N, N)), np.zeros((N, V))
        for ws, ts in sentences:
            ids = [self.t2i[t] for t in ts]
            init[ids[0]] += 1
            for a, b in zip(ids, ids[1:]):
                trans[a, b] += 1
            for w, t in zip(ws, ids):
                emit[t, self.w2i[w.lower()]] += 1

        self.log_pi = np.log((init + k) / (init.sum() + k * N))
        self.log_A  = np.log((trans + k) / (trans.sum(1, keepdims=True) + k * N))
        self.log_B  = np.log((emit + k) / (emit.sum(1, keepdims=True) + k * V))
        return self

    def _obs(self, words):
        unk = self.w2i[UNK]
        return [self.w2i.get(w.lower(), unk) for w in words]

    @staticmethod
    def _logsumexp(x, axis):
        m = x.max(axis=axis, keepdims=True)
        return (m + np.log(np.exp(x - m).sum(axis=axis, keepdims=True))).squeeze(axis)

    def forward(self, words):
        """Returns log P(words) via the forward algorithm."""
        o = self._obs(words)
        alpha = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, len(o)):
            # alpha_t(j) = logsum_i [alpha_{t-1}(i) + log A(i,j)] + log B(j, o_t)
            alpha = self._logsumexp(alpha[:, None] + self.log_A, axis=0) + self.log_B[:, o[t]]
        return self._logsumexp(alpha, axis=0)

    def viterbi(self, words):
        """Returns the most likely tag sequence."""
        o = self._obs(words)
        T, N = len(o), len(self.tags)
        delta = np.zeros((T, N)); psi = np.zeros((T, N), dtype=int)
        delta[0] = self.log_pi + self.log_B[:, o[0]]
        for t in range(1, T):
            scores = delta[t - 1][:, None] + self.log_A
            psi[t]   = scores.argmax(0)
            delta[t] = scores.max(0) + self.log_B[:, o[t]]
        path = [int(delta[-1].argmax())]
        for t in range(T - 1, 0, -1):
            path.append(int(psi[t, path[-1]]))
        return [self.tags[i] for i in reversed(path)]

hmm = HMM(k=K).fit(train)
print("Tags:", hmm.tags, "| Vocab size:", len(hmm.vocab))
print("Initial probs:", dict(zip(hmm.tags, [round(float(x), 3) for x in np.exp(hmm.log_pi)])))

Tags: ['B-NP', 'I-NP', 'O'] | Vocab size: 4256
Initial probs: {'B-NP': 0.657, 'I-NP': 0.0, 'O': 0.343}


## 3. Forward algorithm – probability of each test sentence

In [5]:
log_probs = [hmm.forward(ws) for ws, _ in test]

with open("sentence_probabilities.csv", "w") as f:
    f.write("sentence_no,log_probability,probability\n")
    for i, lp in enumerate(log_probs, 1):
        f.write(f"{i},{lp:.6f},{np.exp(lp):.6e}\n")

for i, lp in enumerate(log_probs, 1):
    print(f"Sentence {i:>3}: log P = {lp:10.4f} | P = {np.exp(lp):.4e}")
print("\nSaved all", len(log_probs), "sentence probabilities to sentence_probabilities.csv")
# (probabilities for long sentences underflow float64 -> the log value is the reliable one)

Sentence   1: log P =  -270.4513 | P = 3.5036e-118
Sentence   2: log P =  -197.2580 | P = 2.1476e-86
Sentence   3: log P =  -219.3152 | P = 5.6576e-96
Sentence   4: log P =  -239.2772 | P = 1.2112e-104
Sentence   5: log P =  -222.8093 | P = 1.7184e-97
Sentence   6: log P =  -254.5387 | P = 2.8526e-111
Sentence   7: log P =  -272.5374 | P = 4.3501e-119
Sentence   8: log P =  -183.3961 | P = 2.2496e-80
Sentence   9: log P =  -208.7755 | P = 2.1377e-91
Sentence  10: log P =  -171.1279 | P = 4.7875e-75
Sentence  11: log P =  -116.0027 | P = 4.1750e-51
Sentence  12: log P =  -170.0405 | P = 1.4202e-74
Sentence  13: log P =  -186.5434 | P = 9.6654e-82
Sentence  14: log P =  -343.3505 | P = 7.6695e-150
Sentence  15: log P =  -158.9058 | P = 9.7292e-70
Sentence  16: log P =   -82.3741 | P = 1.6803e-36
Sentence  17: log P =  -215.0995 | P = 3.8324e-94
Sentence  18: log P =   -85.4887 | P = 7.4601e-38
Sentence  19: log P =  -143.1054 | P = 7.0813e-63
Sentence  20: log P =  -146.3897 | P = 2.6532

## 4. Viterbi decoding + evaluation with `seqeval`

In [6]:
y_true = [ts for _, ts in test]
y_pred = [hmm.viterbi(ws) for ws, _ in test]

print(classification_report(y_true, y_pred, digits=4))
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
print(f"F1       : {f1_score(y_true, y_pred):.4f}")
print(f"Token accuracy: {accuracy_score(y_true, y_pred):.4f}")

              precision    recall  f1-score   support

          NP     0.6754    0.7579    0.7143       475

   micro avg     0.6754    0.7579    0.7143       475
   macro avg     0.6754    0.7579    0.7143       475
weighted avg     0.6754    0.7579    0.7143       475

Precision: 0.6754
Recall   : 0.7579
F1       : 0.7143
Token accuracy: 0.8650


In [7]:
# Sample predictions
for i in range(3):
    ws, gold = test[i]
    print(f"{'word':<15}{'gold':<8}{'pred':<8}")
    for w, g, p in zip(ws, gold, y_pred[i]):
        print(f"{w:<15}{g:<8}{p:<8}{'' if g == p else '  <--'}")
    print()

word           gold    pred    
Confidence     B-NP    B-NP    
in             O       O       
the            B-NP    B-NP    
pound          I-NP    I-NP    
is             O       O       
widely         O       B-NP      <--
expected       O       O       
to             O       O       
take           O       O       
another        B-NP    B-NP    
sharp          I-NP    I-NP    
dive           I-NP    I-NP    
if             O       O       
trade          B-NP    B-NP    
figures        I-NP    I-NP    
for            O       O       
September      B-NP    B-NP    
,              O       O       
due            O       O       
for            O       O       
release        B-NP    B-NP    
tomorrow       B-NP    I-NP      <--
,              O       O       
fail           O       B-NP      <--
to             O       O       
show           O       O       
a              B-NP    B-NP    
substantial    I-NP    I-NP    
improvement    I-NP    I-NP    
from           O       O 

## 5. (Optional) Effect of the Add-K value

In [8]:
for k in [1e-4, 1e-3, 1e-2, 1e-1, 1.0]:
    m = HMM(k=k).fit(train)
    preds = [m.viterbi(ws) for ws, _ in test]
    print(f"K={k:<8} F1={f1_score(y_true, preds):.4f}")

K=0.0001   F1=0.6700
K=0.001    F1=0.6733


K=0.01     F1=0.6819


K=0.1      F1=0.7143


K=1.0      F1=0.6773
